In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import sys

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rest_of_season.config import (
    SEASON,
    CURRENT_WEEK,
    N_SIMULATIONS,
    RANDOM_SEED,
    CURRENT_OUTPUT_DIR
)

TARGET_WEEK = CURRENT_WEEK

WEEKLY_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed" / "weekly"
)

OUTPUT_DIR = CURRENT_OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Season:", SEASON)
print("Projection week:", TARGET_WEEK)
print("Simulations:", N_SIMULATIONS)
print("Project root:", PROJECT_ROOT)
print("ROS output:", OUTPUT_DIR)


Season: 2026
Projection week: 8
Simulations: 50,000
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_08
Season: 2026
Projection week: 8
Simulations: 50000
Project root: C:\Users\efriedman\Desktop\NFL-Season-Projections
ROS output: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_08


In [2]:

schedule_path = (
    WEEKLY_DATA_DIR / f"{SEASON}_schedule_current.parquet"
)
completed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)
strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)
injury_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

required_files = [
    schedule_path,
    completed_path,
    strength_path,
    injury_path
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required pipeline output missing: {path}"
        )

schedule = pd.read_parquet(schedule_path)
completed = pd.read_parquet(completed_path)
strength = pd.read_parquet(strength_path)
injury_strength = pd.read_parquet(injury_path)

print("Schedule:", schedule.shape)
print("Completed:", completed.shape)
print("Strength:", strength.shape)
print("Injury strength:", injury_strength.shape)


Schedule: (272, 46)
Completed: (107, 46)
Strength: (32, 25)
Injury strength: (32, 30)


In [3]:

remaining_games = schedule.loc[
    (schedule["week"] >= TARGET_WEEK)
    & schedule["home_score"].isna()
    & schedule["away_score"].isna()
].copy()

remaining_games = remaining_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert remaining_games["game_id"].is_unique
assert remaining_games["home_team"].notna().all()
assert remaining_games["away_team"].notna().all()

print("Remaining regular-season games:", len(remaining_games))

display(
    remaining_games[
        ["week", "gameday", "away_team", "home_team"]
    ].head(20)
)


Remaining regular-season games: 165


,week,gameday,away_team,home_team
0,8,2026-10-29,CAR,GB
1,8,2026-11-01,ARI,DAL
2,8,2026-11-01,ATL,TB
3,8,2026-11-01,BAL,BUF
4,8,2026-11-01,CLE,PIT
5,8,2026-11-01,IND,JAX
6,8,2026-11-01,KC,DEN
7,8,2026-11-01,LAC,LA
8,8,2026-11-01,LV,NYJ
9,8,2026-11-01,MIN,DET


In [4]:

completed = completed.loc[
    (completed["week"] < TARGET_WEEK)
    & completed["home_score"].notna()
    & completed["away_score"].notna()
].copy()

home_records = completed[
    ["home_team", "home_score", "away_score"]
].rename(columns={
    "home_team": "team",
    "home_score": "points_for",
    "away_score": "points_against"
})

away_records = completed[
    ["away_team", "away_score", "home_score"]
].rename(columns={
    "away_team": "team",
    "away_score": "points_for",
    "home_score": "points_against"
})

team_results = pd.concat(
    [home_records, away_records],
    ignore_index=True
)

team_results["wins"] = (
    team_results["points_for"] > team_results["points_against"]
).astype(int)

team_results["losses"] = (
    team_results["points_for"] < team_results["points_against"]
).astype(int)

team_results["ties"] = (
    team_results["points_for"] == team_results["points_against"]
).astype(int)

records = team_results.groupby("team", as_index=False).agg(
    wins=("wins", "sum"),
    losses=("losses", "sum"),
    ties=("ties", "sum"),
    games_played=("wins", "size"),
    points_for=("points_for", "sum"),
    points_against=("points_against", "sum")
)

records["point_differential"] = (
    records["points_for"] - records["points_against"]
)

display(records.sort_values(
    ["wins", "point_differential"],
    ascending=[False, False]
))


,team,wins,losses,ties,games_played,points_for,points_against,point_differential
28,SF,6,1,0,7,197.0,139.0,58.0
20,MIN,6,0,0,6,125.0,74.0,51.0
15,KC,5,1,0,6,143.0,103.0,40.0
5,CHI,5,2,0,7,180.0,143.0,37.0
3,BUF,5,1,0,6,197.0,171.0,26.0
26,PIT,5,2,0,7,153.0,138.0,15.0
9,DEN,5,2,0,7,144.0,141.0,3.0
7,CLE,5,2,0,7,133.0,146.0,-13.0
14,JAX,4,2,0,6,154.0,104.0,50.0
27,SEA,4,3,0,7,168.0,125.0,43.0


In [5]:

required_strength_cols = [
    "team",
    "weekly_team_strength"
]

required_injury_cols = [
    "team",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

for name, frame, columns in [
    ("strength", strength, required_strength_cols),
    ("injury", injury_strength, required_injury_cols)
]:
    missing = set(columns) - set(frame.columns)
    if missing:
        raise KeyError(
            f"{name} missing columns: {sorted(missing)}"
        )

team_snapshot = strength[
    required_strength_cols
].merge(
    injury_strength[required_injury_cols],
    on="team",
    how="left",
    validate="one_to_one"
).merge(
    records,
    on="team",
    how="left",
    validate="one_to_one"
)

assert len(team_snapshot) == 32
assert team_snapshot["team"].is_unique
assert team_snapshot["weekly_team_strength"].notna().all()

print("Teams:", len(team_snapshot))
display(team_snapshot.head(10))


Teams: 32


,team,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength,wins,losses,ties,games_played,points_for,points_against,point_differential
0,SEA,5.114660,-0.348375,4.766285,4,3,0,7,168.0,125.0,43.0
1,BUF,4.947555,-0.154950,4.792605,5,1,0,6,197.0,171.0,26.0
2,DEN,3.957970,-0.422400,3.535570,5,2,0,7,144.0,141.0,3.0
3,SF,3.931716,0.000000,3.931716,6,1,0,7,197.0,139.0,58.0
4,LA,3.674363,-0.838650,2.835713,3,4,0,7,170.0,159.0,11.0
5,MIN,3.147998,-0.311325,2.836673,6,0,0,6,125.0,74.0,51.0
6,JAX,3.094193,-0.449625,2.644568,4,2,0,6,154.0,104.0,50.0
7,NE,2.959946,-0.894525,2.065421,4,3,0,7,160.0,143.0,17.0
8,DET,2.938408,-1.080225,1.858183,4,2,0,6,184.0,165.0,19.0
9,KC,2.841071,-0.406875,2.434196,5,1,0,6,143.0,103.0,40.0


In [6]:

team_snapshot["season"] = SEASON
team_snapshot["snapshot_week"] = TARGET_WEEK

remaining_games["season"] = SEASON
remaining_games["snapshot_week"] = TARGET_WEEK

team_snapshot.to_parquet(
    OUTPUT_DIR / "team_snapshot.parquet",
    index=False
)

remaining_games.to_parquet(
    OUTPUT_DIR / "remaining_schedule.parquet",
    index=False
)

team_snapshot.to_csv(
    OUTPUT_DIR / "team_snapshot.csv",
    index=False
)

remaining_games[
    ["week", "gameday", "away_team", "home_team"]
].to_csv(
    OUTPUT_DIR / "remaining_schedule.csv",
    index=False
)

print(f"Saved Week {TARGET_WEEK} ROS inputs.")
print("Output directory:", OUTPUT_DIR)
print("Teams:", len(team_snapshot))
print("Remaining games:", len(remaining_games))


Saved Week 8 ROS inputs.
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_08
Teams: 32
Remaining games: 165
